# 1957 Perceptron

Frank Rosenblatt's perceptron is one of the earliest neural network models and a useful starting point for an AI timeline notebook.

## Weights, Bias, and Equations

A perceptron receives one or more input values and combines them into a single score before making a binary decision.

- **Weights** (`weights`) control how strongly each input affects the result. A positive weight pushes the score upward when the input is `1`; a negative weight pushes it downward.
- **Bias** (`bias`) sets the threshold for the decision. Its goal is to make the perceptron more or less strict about when it should return `1`.

The perceptron first calculates the weighted sum:

$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$$

Then it applies a step activation function:

$$\hat{y} = \begin{cases} 1, & z \ge 0 \\ 0, & z < 0 \end{cases}$$

In the code below, `total` is `z`, and `return 1 if total >= 0 else 0` is the step function.

During training, each example has `inputs` and a `desired_output`. The trainer compares the perceptron's prediction with the desired output:

$$error = desired\ output - prediction$$

Then it updates the weights and bias:

$$w_i = w_i + learning\ rate \times error \times x_i$$

$$b = b + learning\ rate \times error$$

In [1]:
class Perceptron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def predict(self, inputs):
        total = sum(weight * value for weight, value in zip(self.weights, inputs)) + self.bias
        return 1 if total >= 0 else 0


class Trainer:
    def __init__(self, perceptron, learning_rate=1):
        self.perceptron = perceptron
        self.learning_rate = learning_rate

    def train(self, training_data, epochs=10):
        for _ in range(epochs):
            mistakes = 0

            for inputs, desired_output in training_data:
                prediction = self.perceptron.predict(inputs)
                error = desired_output - prediction

                if error != 0:
                    self.perceptron.weights = [
                        weight + self.learning_rate * error * value
                        for weight, value in zip(self.perceptron.weights, inputs)
                    ]
                    self.perceptron.bias += self.learning_rate * error
                    mistakes += 1

            if mistakes == 0:
                break

        return self.perceptron


and_training_data = [
    ((0, 0), 0),
    ((0, 1), 0),
    ((1, 0), 0),
    ((1, 1), 1),
]

or_training_data = [
    ((0, 0), 0),
    ((0, 1), 1),
    ((1, 0), 1),
    ((1, 1), 1),
]

and_gate = Perceptron(weights=[0, 0], bias=0)
or_gate = Perceptron(weights=[0, 0], bias=0)

Trainer(and_gate).train(and_training_data)
Trainer(or_gate).train(or_training_data)

In [2]:
binary_inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

print(f"Learned AND parameters: weights={and_gate.weights}, bias={and_gate.bias}")
print("AND gate")
for values in binary_inputs:
    print(f"{values} -> {and_gate.predict(values)}")

print(f"\nLearned OR parameters: weights={or_gate.weights}, bias={or_gate.bias}")
print("OR gate")
for values in binary_inputs:
    print(f"{values} -> {or_gate.predict(values)}")


for inputs, desired_output in and_training_data:
    assert and_gate.predict(inputs) == desired_output

for inputs, desired_output in or_training_data:
    assert or_gate.predict(inputs) == desired_output

print("\nAll perceptron gate checks passed.")

Learned AND parameters: weights=[2, 1], bias=-3
AND gate
(0, 0) -> 0
(0, 1) -> 0
(1, 0) -> 0
(1, 1) -> 1

Learned OR parameters: weights=[1, 1], bias=-1
OR gate
(0, 0) -> 0
(0, 1) -> 1
(1, 0) -> 1
(1, 1) -> 1

All perceptron gate checks passed.


## XOR with a Multilayer Perceptron

A single perceptron cannot learn XOR because XOR is not linearly separable. To solve it, we use a small multilayer perceptron with two inputs, one hidden layer, and one output.

The topology below has its own `MultilayerPerceptronNeuron` class. That keeps the single-layer `Perceptron` focused on step activation, while each multilayer neuron owns its sigmoid activation behavior so the trainer can calculate gradients with backpropagation.

Backpropagation starts at the output layer, measures how far the prediction is from the desired output, and sends that error backward through the hidden layer. Each neuron then updates its weights and bias according to how much it contributed to the error.

For this topology, each neuron uses the sigmoid activation:

$$a = \frac{1}{1 + e^{-z}}$$

The output layer delta is:

$$\delta_{output} = (desired\ output - prediction) \times a(1 - a)$$

Each hidden layer delta uses the deltas from the next layer:

$$\delta_{hidden} = \left(\sum w_{next}\delta_{next}\right) \times a(1 - a)$$

In [3]:
import math
import random


class MultilayerPerceptronNeuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def weighted_sum(self, inputs):
        return sum(weight * value for weight, value in zip(self.weights, inputs)) + self.bias

    def activate(self, inputs):
        return 1 / (1 + math.exp(-self.weighted_sum(inputs)))

    def activation_derivative(self, activation):
        return activation * (1 - activation)


class MultilayerPerceptron:
    def __init__(self, layer_sizes, seed=1):
        if len(layer_sizes) < 2:
            raise ValueError("A multilayer perceptron needs at least an input layer and an output layer.")

        random_generator = random.Random(seed)
        self.layers = []

        for input_count, neuron_count in zip(layer_sizes, layer_sizes[1:]):
            layer = [
                MultilayerPerceptronNeuron(
                    weights=[random_generator.uniform(-1, 1) for _ in range(input_count)],
                    bias=random_generator.uniform(-1, 1),
                )
                for _ in range(neuron_count)
            ]
            self.layers.append(layer)

    def forward(self, inputs):
        activations = [list(inputs)]

        for layer in self.layers:
            current_outputs = [neuron.activate(activations[-1]) for neuron in layer]
            activations.append(current_outputs)

        return activations

    def predict_probability(self, inputs):
        return self.forward(inputs)[-1][0]

    def predict(self, inputs):
        return 1 if self.predict_probability(inputs) >= 0.5 else 0


class MultilayerPerceptronTrainer:
    def __init__(self, topology, learning_rate=0.5):
        self.topology = topology
        self.learning_rate = learning_rate

    def train(self, training_data, epochs=10000, target_error=0.01):
        for _ in range(epochs):
            total_error = 0

            for inputs, desired_output in training_data:
                desired_outputs = [desired_output]
                activations = self.topology.forward(inputs)
                output_activations = activations[-1]
                output_errors = [
                    desired - actual
                    for desired, actual in zip(desired_outputs, output_activations)
                ]
                total_error += sum(error ** 2 for error in output_errors)

                deltas = [None] * len(self.topology.layers)
                deltas[-1] = [
                    error * neuron.activation_derivative(activation)
                    for error, activation, neuron in zip(
                        output_errors,
                        output_activations,
                        self.topology.layers[-1],
                    )
                ]

                for layer_index in range(len(self.topology.layers) - 2, -1, -1):
                    current_layer = self.topology.layers[layer_index]
                    next_layer = self.topology.layers[layer_index + 1]
                    current_activations = activations[layer_index + 1]
                    layer_deltas = []

                    for neuron_index, (neuron, activation) in enumerate(zip(current_layer, current_activations)):
                        downstream_error = sum(
                            next_neuron.weights[neuron_index] * deltas[layer_index + 1][next_index]
                            for next_index, next_neuron in enumerate(next_layer)
                        )
                        layer_deltas.append(downstream_error * neuron.activation_derivative(activation))

                    deltas[layer_index] = layer_deltas

                for layer_index, layer in enumerate(self.topology.layers):
                    previous_activations = activations[layer_index]

                    for neuron, delta in zip(layer, deltas[layer_index]):
                        neuron.weights = [
                            weight + self.learning_rate * delta * activation
                            for weight, activation in zip(neuron.weights, previous_activations)
                        ]
                        neuron.bias += self.learning_rate * delta

            if total_error < target_error:
                break

        return self.topology


xor_training_data = [
    ((0, 0), 0),
    ((0, 1), 1),
    ((1, 0), 1),
    ((1, 1), 0),
]

xor_gate = MultilayerPerceptron(layer_sizes=[2, 2, 1], seed=1)
MultilayerPerceptronTrainer(xor_gate, learning_rate=0.5).train(xor_training_data)

print("XOR gate")
for values, _ in xor_training_data:
    probability = xor_gate.predict_probability(values)
    print(f"{values} -> {xor_gate.predict(values)} (probability={probability:.3f})")

for inputs, desired_output in xor_training_data:
    assert xor_gate.predict(inputs) == desired_output

print("\nXOR multilayer perceptron checks passed.")

XOR gate
(0, 0) -> 0 (probability=0.050)
(0, 1) -> 1 (probability=0.957)
(1, 0) -> 1 (probability=0.940)
(1, 1) -> 0 (probability=0.044)

XOR multilayer perceptron checks passed.
